In [14]:
import os
import sys

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
    
from ytmusic_library import YTMusicPlaylists, PLAYLIST_TSV_COLUMNS


RUN_API_AUTH_TEST = True
HEADER_FILE='../headers_auth.json'
PLAYCOUNT_FILE='../playlists/_ytmusic_lastfm_match_id_map.tsv'
NOT_LIKE_PLAYLIST_TSV ='../playlists/xx not like.tsv'
LIKE_PLAYLIST_TSV = '../playlists/_liked_tracks.tsv'
ALL_TRACKS_TSV = '../playlists/_tracks_db.tsv'
BACKUP_DIR = '../playlists/'


In [15]:
Y = YTMusicPlaylists(header=HEADER_FILE, playcount_map=PLAYCOUNT_FILE,  not_like_tsv=NOT_LIKE_PLAYLIST_TSV)
if RUN_API_AUTH_TEST: Y.test_ytmusic_api()
print(f"Loaded {len(Y.playlists['title'].unique())} playlists")
# res = Y.yt.search(query='Hot Fuss', filter='albums', limit=3)


Using header file: ../headers_auth.json
Loaded 266580 playounts from 104345 tracks
Test Passed in 2.49 seconds
Using ytmusicapi version: 0.25.0
Loaded 512 playlists


## WIP Like not liker miner

### TODO
```
fuzzy track id should be slugified "artist - track"
load Likes transform into fuzzy track id
load not likes transform into fuzzy track id
load all tracks  transform into fuzzy track id

for each Like unique fuzzy album id
 find matches in all tracks
 for each match 
  add to tmp like playlist

for each not Like unique fuzzy album id
 find matches in all tracks
 for each match 
  if match rating is not LIKE
   add to tmp not like playlist

look at results and like or merge with not lik pl
maybe scann pl tsvs for pl with lots ok like or not like and clean up
```

In [37]:
module_path = os.path.abspath(os.path.join('../../music-sources-unified/'))
if module_path not in sys.path:
    sys.path.append(module_path)
import unify_lib as uni
import pandas as pd

# like_df = pd.read_csv(LIKE_PLAYLIST_TSV, sep='\t', index_col=0)
# Concat like playlists together and remove dupes
yt_like_df = pd.read_csv('../playlists/Liked Music.tsv', sep='\t', index_col=0)
yt_like_df = yt_like_df.set_index('videoId', drop=True)
like_df =  pd.read_csv(LIKE_PLAYLIST_TSV, sep='\t', index_col=0)
like_df = pd.concat([like_df, yt_like_df[like_df.columns]])
like_df = like_df[~like_df.index.duplicated(keep='first')]
del yt_like_df

not_like_df = pd.read_csv(NOT_LIKE_PLAYLIST_TSV, sep='\t', index_col=0)
not_like_df = not_like_df.set_index('videoId', drop=True)
all_df = pd.read_csv(ALL_TRACKS_TSV, sep='\t', index_col=0)

print(f'Loaded {len(like_df)} like, {len(not_like_df)} not like entries, and {len(all_df)} total tracks')
not_like_df = not_like_df.loc[not_like_df['likeStatus'] !='LIKE']
print(f'Keeping {len(not_like_df)} not like after remove LIKE')
# Add id
like_df['fuzzy_track_id'] = like_df.apply(uni.make_ytmusic_fuzzy_slugified_track_id, axis=1)
not_like_df['fuzzy_track_id'] = not_like_df.apply(uni.make_ytmusic_fuzzy_slugified_track_id, axis=1)
all_df['fuzzy_track_id'] = all_df.apply(uni.make_ytmusic_fuzzy_slugified_track_id, axis=1)

# Sets for quick lookup
not_like_vids = frozenset(not_like_df.index)
not_like_fuzzy_ids = frozenset(not_like_df['fuzzy_track_id'])
like_vids = frozenset(like_df.index)
like_fuzzy_ids = frozenset(like_df['fuzzy_track_id'])

Loaded 39646 like, 4531 not like entries, and 149039 total tracks
Keeping 4383 not like after remove LIKE


Saving _need_like tsv with 2868 entries
Saving _need_like_but_is_not_like tsv with 187 entries
Saving _need_not_like tsv with 4749 entries
Saving _need_not_like_but_is_like tsv with 259 entries


In [39]:
print('Looking through LIKE tracks (~5m)')
like_impacted_playlists = []
new_likes = set()
skip_not_like = set()
for fuzzy_track_id in like_fuzzy_ids:
  matches = all_df.loc[all_df['fuzzy_track_id'] == fuzzy_track_id]
  if not len(matches): continue
  for match in matches.itertuples():
    if match.likeStatus == 'LIKE': continue
    match_name = f'{match.artist} - {match.album} - {match.title}'
    if match.Index in not_like_vids:
      skip_not_like.add(match.Index)
      continue
    new_likes.add(match.Index)
    if pd.isna(match.playlists): continue
    like_impacted_playlists.append(match.playlists) 
print(f' Found {len(new_likes)} new tracks to LIKE')
print(f' Found {len(skip_not_like)} new tracks to LIKE but they are already in NOT LIKE')

print(100*'*')
print('Looking through NOT LIKE tracks (~1m)')
not_like_impacted_playlists = []
new_not_likes = set()
skip_is_like = set()
for fuzzy_track_id in not_like_fuzzy_ids:
  matches = all_df.loc[all_df['fuzzy_track_id'] == fuzzy_track_id]
  if not len(matches): continue
  for match in matches.itertuples():
    match_name = f'{match.artist} - {match.album} - {match.title}'
    if match.Index in like_vids or match.Index in new_likes :
      skip_is_like.add(match.Index)
      continue
    new_not_likes.add(match.Index)
    if not pd.isna(match.playlists): continue
    not_like_impacted_playlists.append(match.playlists) 
print(f' Found {len(new_not_likes)} new tracks to NOT LIKE')
print(f' Found {len(skip_is_like)} new tracks to NOT LIKE but they are already in LIKE')

def generate_impacted_playlist_df(impacted_playlists):
  df = pd.DataFrame(impacted_playlists, columns=['encoded_list'])
  df['encoded_list'] = df['encoded_list'].dropna().str.replace('[nan]', "['nan']")
  df['decoded_list'] = df['encoded_list'].str.lstrip('[').str.rstrip(']').str.split(', ')
  df = df.explode('decoded_list')
  return df['decoded_list'].str.strip("'")

print(100*'*')
like_impacted_playlists_df = generate_impacted_playlist_df(like_impacted_playlists)
print(f'Top 20 playlists impacted by LIKE: {like_impacted_playlists_df.value_counts().head(20)}')


print(100*'*')

like_not_like_res = {
  '_need_like': all_df.loc[all_df.index.isin(new_likes)],
  '_need_like_but_is_not_like': all_df.loc[all_df.index.isin(skip_not_like)],
  '_need_not_like': all_df.loc[all_df.index.isin(new_not_likes)],
  '_need_not_like_but_is_like': all_df.loc[all_df.index.isin(skip_is_like)],
}

save_cols = ['title',  'artist', 'album', 'albumArtist', 
             'likeStatus',  'averageRating',
             'albumYear', 'albumType','duration_seconds']
for k, v in like_not_like_res.items():
  print(f'Saving {k} tsv with {len(v)} entries')
  v[save_cols].to_csv(os.path.join(BACKUP_DIR, k+'.tsv'), sep='\t', index=True)
print(100*'*')

## TODO for loop like above (or inside that)
# Y.yt.create_playlist(
#   title='_likes_new', video_ids=list(new_likes), privacy_status='PRIVATE',
#   description=f'{len(new_likes)} tracks that should probably be like')
# Y.yt.create_playlist(
#   title='_likes_maybe_new', video_ids=list(skip_not_like), privacy_status='PRIVATE',
#   description=f'{len(skip_not_like)} tracks might be like')
# Y.yt.create_playlist(
#   title='_not_likes_new', video_ids=list(new_not_likes), privacy_status='PRIVATE',
#   description=f'{len(new_not_likes)} tracks that should probably be not like')
# Y.yt.create_playlist(
#   title='_not_likes_maybe_new', video_ids=list(skip_is_like), privacy_status='PRIVATE',
#   description=f'{len(skip_is_like)} tracks might be not like')
# print('Generated playlists')



Looking through LIKE tracks (~5m)
 Found 2868 new tracks to LIKE
 Found 187 new tracks to LIKE but they are already in NOT LIKE
****************************************************************************************************
Looking through NOT LIKE tracks (~1m)
 Found 4749 new tracks to NOT LIKE
 Found 259 new tracks to NOT LIKE but they are already in LIKE
****************************************************************************************************
Top 20 playlists impacted by LIKE: decoded_list
nan                            400
rock 1950s roots radio          86
Jukebox Vintage Party radio     85
y_2012_thumbs_up                62
zzzz_all                        62
soul radio                      52
zzzz_all 4                      47
Chill Supermix                  44
r.hiphop                        43
blues radio                     41
zz__seed_albums                 37
zzzz_all 1                      37
zzzz_all 5                      35
beats                          

```
Looking through LIKE tracks (~5m)
 Found 2724 new tracks to LIKE
 Found 172 new tracks to LIKE but they are already in NOT LIKE
****************************************************************************************************
Looking through NOT LIKE tracks (~1m)
 Found 4691 new tracks to NOT LIKE
 Found 242 new tracks to NOT LIKE but they are already in LIKE
```

In [ ]:
## OLDER
# # also make pl for track ids in _like_tsv but not rated LIKE, these can quickly be inspeced, then regen other above pls
# all_df_like = all_df.loc[all_df.index.isin(like_vids)]
# all_df_like_not_liked = all_df_like.loc[all_df_like['likeStatus'] != 'LIKE']
# all_df_like_not_liked_vids = frozenset(all_df_like_not_liked.index)
# Y.yt.create_playlist(
#   title='_likes_should_be_liked', video_ids=list(all_df_like_not_liked_vids), privacy_status='PRIVATE',
#   description=f'{len(all_df_like_not_liked_vids)} tracks are already marked as like')
# likes_new_not_previously_liked = frozenset(new_likes - all_df_like_not_liked_vids)
# Y.yt.create_playlist(
#   title='_likes_new_not_previously_liked', video_ids=list(likes_new_not_previously_liked), privacy_status='PRIVATE',
#   description=f'{len(likes_new_not_previously_liked)} tracks that should probably be like')

## Split mixed rating playlist into LIKE and INDIFFERENT


In [ ]:
# For each playlist, Create Unrated and Liked Subset Playlist, delete original

playlist_names = [
    'Indie dark side radio',
    'Indie Dreams of Fall radio',
    'Indie Folk radio',
    'indie loose live chill radio',
    'Indie radio',
    'x_r.chillwave_tracks_radio',
    'x_r.indie_tracks_radio',
    'x_r.IndieFolk_tracks_radio',
    'x_r.indierock_tracks_radio',
    'jazz essential radio',
    'Jazz Feels the Blues radio',
    'jazz for spies radio',
    'jazz gloom smooth radio',
    'jazz guitar radio',
    'jazz ken burns radio',
    'jazz radio',
    'jazz Ted Gioia’s How to Listen to Jazz radio',
    'jazz traditional radio',
    'x_r.jazz_tracks_radio',
    'x_r.jazznoir_tracks_radio',
    'x_r.VintageObscura_tracks_radio',
    'x_r.krautrock_tracks_radio',
    'x_r.NewWave_tracks_radio',
    'x_r.PsychedelicRock_tracks_radio',
    'x_r.stonerrock_tracks_radio',
    'x_r.90sAlternative_tracks_radio',
    'garage rock radio',
    'rock 1950s roots radio',
    'rock 1960s classic radio',
    'soul 1960s radio',
    'Soul Classic Sunshine radio',
    'Soul Food Kitchen radio',
    'soul motown radio',
    'soul radio',
    'punk 1970s British radio',
    'punk 1970s radio',
    'Post-Punk 1970s-1980s radio',
    'x_r.PunkRock_tracks_radio',
    'x_r.PostRock_tracks_radio',
    'x_r.90sPunk_tracks_radio',
]


for playlist_name in playlist_names:
    pl_info = Y.playlist_get_info(Y.query_by_title(playlist_name).playlistId)
    Y.move_likes_from_radio_playlist( ########################################
        pl_info, min_n_like=1, verbose=True, playcount_sort=True, ignore_banned=False)

## Update Not Like tsv

In [7]:
not_like_name = os.path.splitext(os.path.basename(NOT_LIKE_PLAYLIST_TSV))[0]
not_like_pl = Y.playlists.loc[Y.playlists['title'] == not_like_name].iloc[0]
not_like_tracks, _ = Y.parse_playlist(Y.playlist_get_info(not_like_pl['playlistId'], use_cache=True))
not_like_tracks = not_like_tracks[PLAYLIST_TSV_COLUMNS].sort_values(['likeStatus', 'artist'], ascending=False)
not_like_tracks.to_csv(os.path.join(BACKUP_DIR, f'{not_like_name}.tsv'), sep='\t', header=True)

## Upload playlist from tsv backup

In [5]:
tsv = '../playlists/x_r.FunkSouMusic_tracks_like.tsv'
Y.playlist_from_tsv(tsv, ignore_banned=True, sort_by_index=True)


Generating x_r.FunkSouMusic_tracks_like ytmusic playlist for 49 tracks
Saved 49 x_r.FunkSouMusic_tracks_like tracks playlist with id: PLWptjpDqazOypETiPz5QXpvMDnJkm08jp


## Query

In [ ]:
Y.query_by_title(playlist_name)

## Get Playlist Counts

In [4]:
playlist_file = '../playlists/_playlist_radio_counts.tsv'
playlists = Y.get_playlist_counts(verbose=False, filter_title='radio')
playlists.loc[playlists.title.str.contains('radio')].sort_values('track_count').to_csv(playlist_file, sep='\t', index=False)
# last run 6-2023

0: Skipping: Liked Music, filter: radio
1: Skipping: 2022 Recap, filter: radio
2: Skipping: Acoustic Guitar Explorations, filter: radio
3: Skipping: ambiant electro, filter: radio
4: Skipping: ambient, filter: radio
5: Skipping: ambient BOC, filter: radio
6: Skipping: ambient classic, filter: radio
7: Skipping: ambient Dream Pop Deep Sleep, filter: radio
8: Skipping: ambient haunting harmonious, filter: radio
9: Skipping: ambient Indie synths, filter: radio
10: {'title': 'ambient Indie Synths radio', 'playlist_id': 'PLWptjpDqazOw7al5TJVXbaiQr1ILDBk_y', 'track_count': 37, 'privacy': 'PRIVATE', 'duration_hours': 3}
11: Skipping: ambient lynchian like, filter: radio
12: {'title': 'ambient lynchian radio', 'playlist_id': 'PLWptjpDqazOy3-1UzBO47DaFjgfzHUwdP', 'track_count': 51, 'privacy': 'PRIVATE', 'duration_hours': 4}
13: Skipping: ambient modern like, filter: radio
14: {'title': 'ambient modern radio', 'playlist_id': 'PLWptjpDqazOwO_d5ayc93-4pTuls5NCxG', 'track_count': 73, 'privacy': 'PR

## Get Public Playlists

In [7]:
Y.get_playlists_by_privacy(privacy='PUBLIC')
# last run 6-2023, 3 public playlists

Found public playlist named: Liked Music
Found public playlist named: Chill Supermix
Found public playlist named: Episodes for Later


title                                                Liked Music
playlistId                                                    LM
thumbnails     [{'url': 'https://www.gstatic.com/youtube/medi...
description                                        Auto playlist
count                                                        NaN
author                                                       NaN
title                                             Chill Supermix
playlistId           RDTMAK5uy_nzfwl2UYv7htL7wDoxbX8Pp6UAFBd92cQ
thumbnails     [{'url': 'https://music.youtube.com/image/mixa...
description                                        YouTube Music
count                                                        NaN
author                                                       NaN
title                                         Episodes for Later
playlistId                                                    SE
thumbnails     [{'url': 'https://www.gstatic.com/youtube/medi...
description              

### Generate playlist froma a list of albums

In [ ]:
name = 'y_2022_albums_to_listen_to'
desc = 'manually selected albums to top off 2022 albums'
albums_to_add = [
    "SZA - SOS",
    "Perfume Genius - Ugly Season",
    "Arctic Monkeys - The Car",
    "The Beths - Expert in a Dying Field",
    "alt-J - The Dream",
     "Soichi Terada (寺田創一) - Asakusa Light",
    "Everything Everything - Raw Data Feel",
    "The Mountain Goats - Bleed Out",
    "Metric - Formentera",
    "Orville Peck - Bronco",
    "Hurray for the Riff Raff - Life on Earth",
    "Freddie Gibbs - $oul $old $eparately",
    "Tove Lo - Dirt Femme",
    "Oren Ambarchi - Shebang",
    "Hatchie - Giving the World Away",
    "Florence + the Machine - Dance Fever",
    "Richard Dawson - The Ruby Cord",
    "Charlotte Adigéry & Bolis Pupul - Topical Dancer",
    "Phoenix (FR) - Alpha Zulu",
    "Elder - Innate Passage",
    "The Black Angels - Wilderness of Mirrors",
    "Black Flower - Magma",
     "Bad Bunny - Un Verano Sin Ti",
     "Red Hot Chili Peppers - Return Of The Dream Canteen",

]

track_ids = []
for a in albums_to_add:
    match = {}
    res = Y.yt.search(query=a, filter='albums', limit=1)
    result_album = f"{res[0]['artists'][0]['name']} - {res[0]['title']}"
    if len(res) == 0 or res[0].get('browseId') == None:
        print(f'Skipping query: {a} bad result: {res}')
        continue
    yt_album = Y.yt.get_album(res[0]['browseId'])
    for t in yt_album['tracks']:
        track_ids.append(t['videoId'])
    print(f'Added {len(yt_album["tracks"])} tracks":\n\tq: {a}\n\tr: {result_album}')

pl_id = Y.yt.create_playlist( title=name,  description=desc, video_ids=track_ids)
print(f'Generated playlist: {name} with id {pl_id}')